# FastConv: transformada e inversa sem acumulador

Este notebook calcula `D = C^T d C`, depois `S = D # G`, e mostra duas formas equivalentes de fazer a inversa `s = A^T S A`.

In [1]:
import numpy as np
from fractions import Fraction

C = np.array([
    [-1, 0,  0,  0],
    [ 0, 1, -1, -1],
    [ 1, 1,  1,  0],
    [ 0, 0,  0,  1],
], dtype=int)

d = np.array([
    [ 0,  1,  2,  3],
    [ 4,  5,  6,  7],
    [ 8,  9, 10, 11],
    [12, 13, 14, 15],
], dtype=int)

G = np.array([
    [Fraction(0),     Fraction(-3, 2), Fraction(-1, 2), Fraction(-2)],
    [Fraction(-9, 2), Fraction(9),     Fraction(3),     Fraction(15, 2)],
    [Fraction(-3, 2), Fraction(3),     Fraction(1),     Fraction(5, 2)],
    [Fraction(-6),    Fraction(21, 2), Fraction(7, 2),  Fraction(8)],
], dtype=object)

A = np.array([
    [1,  0],
    [1,  1],
    [1, -1],
    [0,  1],
], dtype=int)

print("C =")
print(C)
print("\nd =")
print(d)
print("\nG =")
print(G)
print("\nA =")
print(A)


C =
[[-1  0  0  0]
 [ 0  1 -1 -1]
 [ 1  1  1  0]
 [ 0  0  0  1]]

d =
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]

G =
[[Fraction(0, 1) Fraction(-3, 2) Fraction(-1, 2) Fraction(-2, 1)]
 [Fraction(-9, 2) Fraction(9, 1) Fraction(3, 1) Fraction(15, 2)]
 [Fraction(-3, 2) Fraction(3, 1) Fraction(1, 1) Fraction(5, 2)]
 [Fraction(-6, 1) Fraction(21, 2) Fraction(7, 2) Fraction(8, 1)]]

A =
[[ 1  0]
 [ 1  1]
 [ 1 -1]
 [ 0  1]]


In [2]:
def dot_expression(a, b):
    return " + ".join(f"({x})*({y})" for x, y in zip(a, b))


## Multiplicacao basica: D' = C^T d

Nesta primeira etapa, a matriz fixa `C^T` multiplica a matriz variavel `d`. O primeiro laco percorre as colunas de `d`; para cada coluna de `d`, ela e multiplicada pelas linhas de `C^T`, gerando uma coluna completa de `D'`.

In [3]:
D_prime_basic = np.zeros((4, 4), dtype=int)

for col in range(d.shape[1]):
    d_col = d[:, col]

    print("=" * 70)
    print(f"Coluna {col} de d:")
    print(d_col)
    print(f"\nCalculando D'[:, {col}] = C^T d[:, {col}]")

    for row in range(C.T.shape[0]):
        C_T_row = C.T[row, :]
        D_prime_basic[row, col] = C_T_row @ d_col
        expr = dot_expression(C_T_row, d_col)
        print(f"D'[{row}, {col}] = {expr} = {D_prime_basic[row, col]}")

    print(f"\nD'[:, {col}] = {D_prime_basic[:, col].tolist()}")
    print()

print("D' = C^T d =")
print(D_prime_basic)

assert np.array_equal(D_prime_basic, C.T @ d)


Coluna 0 de d:
[ 0  4  8 12]

Calculando D'[:, 0] = C^T d[:, 0]
D'[0, 0] = (-1)*(0) + (0)*(4) + (1)*(8) + (0)*(12) = 8
D'[1, 0] = (0)*(0) + (1)*(4) + (1)*(8) + (0)*(12) = 12
D'[2, 0] = (0)*(0) + (-1)*(4) + (1)*(8) + (0)*(12) = 4
D'[3, 0] = (0)*(0) + (-1)*(4) + (0)*(8) + (1)*(12) = 8

D'[:, 0] = [8, 12, 4, 8]

Coluna 1 de d:
[ 1  5  9 13]

Calculando D'[:, 1] = C^T d[:, 1]
D'[0, 1] = (-1)*(1) + (0)*(5) + (1)*(9) + (0)*(13) = 8
D'[1, 1] = (0)*(1) + (1)*(5) + (1)*(9) + (0)*(13) = 14
D'[2, 1] = (0)*(1) + (-1)*(5) + (1)*(9) + (0)*(13) = 4
D'[3, 1] = (0)*(1) + (-1)*(5) + (0)*(9) + (1)*(13) = 8

D'[:, 1] = [8, 14, 4, 8]

Coluna 2 de d:
[ 2  6 10 14]

Calculando D'[:, 2] = C^T d[:, 2]
D'[0, 2] = (-1)*(2) + (0)*(6) + (1)*(10) + (0)*(14) = 8
D'[1, 2] = (0)*(2) + (1)*(6) + (1)*(10) + (0)*(14) = 16
D'[2, 2] = (0)*(2) + (-1)*(6) + (1)*(10) + (0)*(14) = 4
D'[3, 2] = (0)*(2) + (-1)*(6) + (0)*(10) + (1)*(14) = 8

D'[:, 2] = [8, 16, 4, 8]

Coluna 3 de d:
[ 3  7 11 15]

Calculando D'[:, 3] = C^T d[:, 3]

## Transformada: D = C^T d C

Primeiro calcula-se `D' = C^T d` por coluna de `d`: para cada coluna de `d`, ela e multiplicada pelas linhas de `C^T`, formando uma coluna de `D'`. Depois calcula-se `D = D' C`.

In [4]:
D_prime = np.zeros((4, 4), dtype=int)

for col in range(d.shape[1]):
    d_col = d[:, col]

    print("=" * 70)
    print(f"Coluna {col} de d:")
    print(d_col)
    print(f"\n1) Calculando D'[:, {col}] = C^T d[:, {col}]")
    for row in range(C.T.shape[0]):
        C_T_row = C.T[row, :]
        D_prime[row, col] = C_T_row @ d_col
        expr = dot_expression(C_T_row, d_col)
        print(f"D'[{row}, {col}] = {expr} = {D_prime[row, col]}")

    print(f"\nD'[:, {col}] = {D_prime[:, col].tolist()}")
    print()


Coluna 0 de d:
[ 0  4  8 12]

1) Calculando D'[:, 0] = C^T d[:, 0]
D'[0, 0] = (-1)*(0) + (0)*(4) + (1)*(8) + (0)*(12) = 8
D'[1, 0] = (0)*(0) + (1)*(4) + (1)*(8) + (0)*(12) = 12
D'[2, 0] = (0)*(0) + (-1)*(4) + (1)*(8) + (0)*(12) = 4
D'[3, 0] = (0)*(0) + (-1)*(4) + (0)*(8) + (1)*(12) = 8

D'[:, 0] = [8, 12, 4, 8]

Coluna 1 de d:
[ 1  5  9 13]

1) Calculando D'[:, 1] = C^T d[:, 1]
D'[0, 1] = (-1)*(1) + (0)*(5) + (1)*(9) + (0)*(13) = 8
D'[1, 1] = (0)*(1) + (1)*(5) + (1)*(9) + (0)*(13) = 14
D'[2, 1] = (0)*(1) + (-1)*(5) + (1)*(9) + (0)*(13) = 4
D'[3, 1] = (0)*(1) + (-1)*(5) + (0)*(9) + (1)*(13) = 8

D'[:, 1] = [8, 14, 4, 8]

Coluna 2 de d:
[ 2  6 10 14]

1) Calculando D'[:, 2] = C^T d[:, 2]
D'[0, 2] = (-1)*(2) + (0)*(6) + (1)*(10) + (0)*(14) = 8
D'[1, 2] = (0)*(2) + (1)*(6) + (1)*(10) + (0)*(14) = 16
D'[2, 2] = (0)*(2) + (-1)*(6) + (1)*(10) + (0)*(14) = 4
D'[3, 2] = (0)*(2) + (-1)*(6) + (0)*(10) + (1)*(14) = 8

D'[:, 2] = [8, 16, 4, 8]

Coluna 3 de d:
[ 3  7 11 15]

1) Calculando D'[:, 3] =

In [5]:
D = np.zeros((4, 4), dtype=int)

for row in range(D_prime.shape[0]):
    D_prime_row = D_prime[row, :]
    D_row = D_prime_row @ C
    D[row, :] = D_row

    print("=" * 70)
    print(f"Linha {row} de D':")
    print(D_prime_row)
    print(f"\n2) Calculando D[{row}, :] = D'[{row}, :] C")
    for col in range(C.shape[1]):
        expr = dot_expression(D_prime_row, C[:, col])
        print(f"D[{row}, {col}] = {expr} = {D_row[col]}")

    print(f"\nD[{row}, :] = {D_row.tolist()}")
    print()

print("D' = C^T d =")
print(D_prime)
print("\nD = D' C = C^T d C =")
print(D)

assert np.array_equal(D, C.T @ d @ C)


Linha 0 de D':
[8 8 8 8]

2) Calculando D[0, :] = D'[0, :] C
D[0, 0] = (8)*(-1) + (8)*(0) + (8)*(1) + (8)*(0) = 0
D[0, 1] = (8)*(0) + (8)*(1) + (8)*(1) + (8)*(0) = 16
D[0, 2] = (8)*(0) + (8)*(-1) + (8)*(1) + (8)*(0) = 0
D[0, 3] = (8)*(0) + (8)*(-1) + (8)*(0) + (8)*(1) = 0

D[0, :] = [0, 16, 0, 0]

Linha 1 de D':
[12 14 16 18]

2) Calculando D[1, :] = D'[1, :] C
D[1, 0] = (12)*(-1) + (14)*(0) + (16)*(1) + (18)*(0) = 4
D[1, 1] = (12)*(0) + (14)*(1) + (16)*(1) + (18)*(0) = 30
D[1, 2] = (12)*(0) + (14)*(-1) + (16)*(1) + (18)*(0) = 2
D[1, 3] = (12)*(0) + (14)*(-1) + (16)*(0) + (18)*(1) = 4

D[1, :] = [4, 30, 2, 4]

Linha 2 de D':
[4 4 4 4]

2) Calculando D[2, :] = D'[2, :] C
D[2, 0] = (4)*(-1) + (4)*(0) + (4)*(1) + (4)*(0) = 0
D[2, 1] = (4)*(0) + (4)*(1) + (4)*(1) + (4)*(0) = 8
D[2, 2] = (4)*(0) + (4)*(-1) + (4)*(1) + (4)*(0) = 0
D[2, 3] = (4)*(0) + (4)*(-1) + (4)*(0) + (4)*(1) = 0

D[2, :] = [0, 8, 0, 0]

Linha 3 de D':
[8 8 8 8]

2) Calculando D[3, :] = D'[3, :] C
D[3, 0] = (8)*(-1) + (8)

## Multiplicacao ponto a ponto: S = D # G

In [6]:
S = D.astype(object) * G

for i in range(D.shape[0]):
    for j in range(D.shape[1]):
        print(f"S[{i}, {j}] = D[{i}, {j}] * G[{i}, {j}] = ({D[i, j]})*({G[i, j]}) = {S[i, j]}")

print("\nS =")
print(S)


S[0, 0] = D[0, 0] * G[0, 0] = (0)*(0) = 0
S[0, 1] = D[0, 1] * G[0, 1] = (16)*(-3/2) = -24
S[0, 2] = D[0, 2] * G[0, 2] = (0)*(-1/2) = 0
S[0, 3] = D[0, 3] * G[0, 3] = (0)*(-2) = 0
S[1, 0] = D[1, 0] * G[1, 0] = (4)*(-9/2) = -18
S[1, 1] = D[1, 1] * G[1, 1] = (30)*(9) = 270
S[1, 2] = D[1, 2] * G[1, 2] = (2)*(3) = 6
S[1, 3] = D[1, 3] * G[1, 3] = (4)*(15/2) = 30
S[2, 0] = D[2, 0] * G[2, 0] = (0)*(-3/2) = 0
S[2, 1] = D[2, 1] * G[2, 1] = (8)*(3) = 24
S[2, 2] = D[2, 2] * G[2, 2] = (0)*(1) = 0
S[2, 3] = D[2, 3] * G[2, 3] = (0)*(5/2) = 0
S[3, 0] = D[3, 0] * G[3, 0] = (0)*(-6) = 0
S[3, 1] = D[3, 1] * G[3, 1] = (16)*(21/2) = 168
S[3, 2] = D[3, 2] * G[3, 2] = (0)*(7/2) = 0
S[3, 3] = D[3, 3] * G[3, 3] = (0)*(8) = 0

S =
[[Fraction(0, 1) Fraction(-24, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(-18, 1) Fraction(270, 1) Fraction(6, 1) Fraction(30, 1)]
 [Fraction(0, 1) Fraction(24, 1) Fraction(0, 1) Fraction(0, 1)]
 [Fraction(0, 1) Fraction(168, 1) Fraction(0, 1) Fraction(0, 1)]]


## Inversa 1: primeiro S' = S A, depois s = A^T S'

Primeiro faca `S' = S A`, fazendo uma linha de `S` com uma coluna de `A` ate formar uma coluna de `S'`. Depois faca `s = A^T S'`, pegando essa coluna e multiplicando pelas linhas de `A^T`.

In [7]:
S_prime_right = np.zeros((4, 2), dtype=object)
s_right = np.zeros((2, 2), dtype=object)

for j in range(A.shape[1]):
    S_prime_col = S @ A[:, j]
    s_col = A.T @ S_prime_col

    S_prime_right[:, j] = S_prime_col
    s_right[:, j] = s_col

    print("=" * 70)
    print(f"Coluna {j} de A:")
    print(A[:, j])
    print(f"\n1) Calculando S'[:, {j}] = S A[:, {j}]")
    for row in range(S.shape[0]):
        expr = dot_expression(S[row, :], A[:, j])
        print(f"S'[{row}, {j}] = {expr} = {S_prime_col[row]}")

    print(f"\nS'[:, {j}] = {S_prime_col.tolist()}")
    print(f"\n2) Calculando s[:, {j}] = A^T S'[:, {j}]")
    for row in range(A.T.shape[0]):
        expr = dot_expression(A.T[row, :], S_prime_col)
        print(f"s[{row}, {j}] = {expr} = {s_col[row]}")

    print(f"\ns[:, {j}] = {s_col.tolist()}")
    print()


Coluna 0 de A:
[1 1 1 0]

1) Calculando S'[:, 0] = S A[:, 0]
S'[0, 0] = (0)*(1) + (-24)*(1) + (0)*(1) + (0)*(0) = -24
S'[1, 0] = (-18)*(1) + (270)*(1) + (6)*(1) + (30)*(0) = 258
S'[2, 0] = (0)*(1) + (24)*(1) + (0)*(1) + (0)*(0) = 24
S'[3, 0] = (0)*(1) + (168)*(1) + (0)*(1) + (0)*(0) = 168

S'[:, 0] = [Fraction(-24, 1), Fraction(258, 1), Fraction(24, 1), Fraction(168, 1)]

2) Calculando s[:, 0] = A^T S'[:, 0]
s[0, 0] = (1)*(-24) + (1)*(258) + (1)*(24) + (0)*(168) = 258
s[1, 0] = (0)*(-24) + (1)*(258) + (-1)*(24) + (1)*(168) = 402

s[:, 0] = [Fraction(258, 1), Fraction(402, 1)]

Coluna 1 de A:
[ 0  1 -1  1]

1) Calculando S'[:, 1] = S A[:, 1]
S'[0, 1] = (0)*(0) + (-24)*(1) + (0)*(-1) + (0)*(1) = -24
S'[1, 1] = (-18)*(0) + (270)*(1) + (6)*(-1) + (30)*(1) = 294
S'[2, 1] = (0)*(0) + (24)*(1) + (0)*(-1) + (0)*(1) = 24
S'[3, 1] = (0)*(0) + (168)*(1) + (0)*(-1) + (0)*(1) = 168

S'[:, 1] = [Fraction(-24, 1), Fraction(294, 1), Fraction(24, 1), Fraction(168, 1)]

2) Calculando s[:, 1] = A^T S'[:,

In [8]:
print("S' = S A =")
print(S_prime_right)
print("\ns = A^T S' = A^T S A =")
print(s_right)

assert np.array_equal(s_right, A.T @ S @ A)


S' = S A =
[[Fraction(-24, 1) Fraction(-24, 1)]
 [Fraction(258, 1) Fraction(294, 1)]
 [Fraction(24, 1) Fraction(24, 1)]
 [Fraction(168, 1) Fraction(168, 1)]]

s = A^T S' = A^T S A =
[[Fraction(258, 1) Fraction(294, 1)]
 [Fraction(402, 1) Fraction(438, 1)]]


## Inversa 2: primeiro S' = A^T S, depois s = S' A

Primeiro faca `S' = A^T S`, fazendo uma linha de `A^T` com uma coluna de `S` ate formar uma linha de `S'`. Depois faca `s = S' A`, pegando essa linha e multiplicando pelas colunas de `A`.

In [9]:
S_prime_left = np.zeros((2, 4), dtype=object)
s_left = np.zeros((2, 2), dtype=object)

for row in range(A.T.shape[0]):
    S_prime_row = A.T[row, :] @ S
    s_row = S_prime_row @ A

    S_prime_left[row, :] = S_prime_row
    s_left[row, :] = s_row

    print("=" * 70)
    print(f"Linha {row} de A^T:")
    print(A.T[row, :])
    print(f"\n1) Calculando S'[{row}, :] = A^T[{row}, :] S")
    for col in range(S.shape[1]):
        expr = dot_expression(A.T[row, :], S[:, col])
        print(f"S'[{row}, {col}] = {expr} = {S_prime_row[col]}")

    print(f"\nS'[{row}, :] = {S_prime_row.tolist()}")
    print(f"\n2) Calculando s[{row}, :] = S'[{row}, :] A")
    for col in range(A.shape[1]):
        expr = dot_expression(S_prime_row, A[:, col])
        print(f"s[{row}, {col}] = {expr} = {s_row[col]}")

    print(f"\ns[{row}, :] = {s_row.tolist()}")
    print()


Linha 0 de A^T:
[1 1 1 0]

1) Calculando S'[0, :] = A^T[0, :] S
S'[0, 0] = (1)*(0) + (1)*(-18) + (1)*(0) + (0)*(0) = -18
S'[0, 1] = (1)*(-24) + (1)*(270) + (1)*(24) + (0)*(168) = 270
S'[0, 2] = (1)*(0) + (1)*(6) + (1)*(0) + (0)*(0) = 6
S'[0, 3] = (1)*(0) + (1)*(30) + (1)*(0) + (0)*(0) = 30

S'[0, :] = [Fraction(-18, 1), Fraction(270, 1), Fraction(6, 1), Fraction(30, 1)]

2) Calculando s[0, :] = S'[0, :] A
s[0, 0] = (-18)*(1) + (270)*(1) + (6)*(1) + (30)*(0) = 258
s[0, 1] = (-18)*(0) + (270)*(1) + (6)*(-1) + (30)*(1) = 294

s[0, :] = [Fraction(258, 1), Fraction(294, 1)]

Linha 1 de A^T:
[ 0  1 -1  1]

1) Calculando S'[1, :] = A^T[1, :] S
S'[1, 0] = (0)*(0) + (1)*(-18) + (-1)*(0) + (1)*(0) = -18
S'[1, 1] = (0)*(-24) + (1)*(270) + (-1)*(24) + (1)*(168) = 414
S'[1, 2] = (0)*(0) + (1)*(6) + (-1)*(0) + (1)*(0) = 6
S'[1, 3] = (0)*(0) + (1)*(30) + (-1)*(0) + (1)*(0) = 30

S'[1, :] = [Fraction(-18, 1), Fraction(414, 1), Fraction(6, 1), Fraction(30, 1)]

2) Calculando s[1, :] = S'[1, :] A
s[1, 0

In [10]:
print("S' = A^T S =")
print(S_prime_left)
print("\ns = S' A = A^T S A =")
print(s_left)

assert np.array_equal(s_left, s_right)
print("\nVerificacao: as duas formas da inversa geram o mesmo s.")


S' = A^T S =
[[Fraction(-18, 1) Fraction(270, 1) Fraction(6, 1) Fraction(30, 1)]
 [Fraction(-18, 1) Fraction(414, 1) Fraction(6, 1) Fraction(30, 1)]]

s = S' A = A^T S A =
[[Fraction(258, 1) Fraction(294, 1)]
 [Fraction(402, 1) Fraction(438, 1)]]

Verificacao: as duas formas da inversa geram o mesmo s.
